# 🎓 Stage 3-1: 파인튜닝 실행 (주석판)

### `refined_law_qa.json` (3,000건) 으로 EXAONE 4.0 1.2B 파인튜닝

> **출력**: `models/v1_law_adapter/` (LoRA 어댑터)  
> **다음**: `02_merge_model.ipynb` 에서 병합  
> **소요**: RTX 3080 기준 약 20~40분

---

## 📌 이 노트북의 전체 그림

```
[원본 EXAONE 4.0 1.2B]
        │
        ├─ 4-bit 양자화 로드 (QLoRA)
        │
        └─ + LoRA 어댑터 부착 (학습 가능한 소수의 파라미터만)
                │
                ├─ refined_law_qa.json (3,000건) 으로 학습
                │
                └─ models/v1_law_adapter/ 로 저장
                        │
                        ▼
                (다음 노트북에서 베이스 모델과 병합)
```

**QLoRA 핵심 아이디어**: 베이스 모델은 4-bit로 얼려두고, 작은 LoRA 어댑터(r=8)만 학습 → VRAM ↓↓, 학습 속도 ↑↑

---

## ⚠️ 사전 조건
- [ ] `data/refined_law_qa.json` 존재 (3,000건 Alpaca 포맷)
- [ ] HUGGINGFACE_TOKEN 설정 (`.env`)
- [ ] VRAM 5GB 이상
- [ ] 이 노트북은 **프로젝트 루트에서 실행** 되어야 함 (정확히는 `notebooks/` 아래)


---
## 1️⃣ 환경 확인

**왜 하는가?**  
- 파인튜닝은 **버전 호환성에 매우 민감**하다. 특히 EXAONE은 `transformers >= 4.54.0` 필수.
- GPU/VRAM을 미리 확인해 학습 도중 OOM으로 죽는 일을 방지.
- 프로젝트 루트를 `sys.path`에 넣어 `config.settings` import를 가능하게 함.

In [1]:
import sys, os

# 1-1. 프로젝트 루트를 sys.path에 추가 (config import용)
#      notebooks/ 에서 실행한다고 가정하므로 상위 디렉토리(..)가 루트
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# 1-2. 핵심 라이브러리 임포트
import torch, transformers, peft, trl

# 1-3. 프로젝트 설정값 로드 (하드코딩 방지 → 실험 재현성 확보)
from config.settings import (
    BASE_MODEL_NAME,      # 베이스 모델 경로/이름
    TRAIN_DATA_PATH,      # 학습 데이터 JSON 경로
    ADAPTER_DIR,          # LoRA 어댑터 저장 폴더
    SYSTEM_LEGAL,         # 법률 도메인 시스템 프롬프트
    MAX_SEQ_LENGTH,       # 토큰 최대 길이 (1024 등)
    LORA_CONFIG,          # r, alpha, dropout, target_modules
    TRAIN_CONFIG,         # epochs, batch, lr, optim 등
    print_settings,       # 설정 출력 헬퍼
)

print(f"✅ PyTorch:      {torch.__version__}")
print(f"✅ transformers: {transformers.__version__}")
print(f"✅ peft:         {peft.__version__}")
print(f"✅ GPU:          {torch.cuda.get_device_name(0)}")
print(f"✅ VRAM:         {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# 1-4. transformers 버전 가드
#      EXAONE 아키텍처는 4.54.0 이상에서 공식 지원
major, minor = map(int, transformers.__version__.split(".")[:2])
assert (major, minor) >= (4, 54), \
    f"transformers >= 4.54.0 필요 (현재 {transformers.__version__})"

print()
print_settings()  # 이번 실험의 모든 설정을 깔끔하게 출력

✅ PyTorch:      2.5.1+cu121
✅ transformers: 5.5.4
✅ peft:         0.19.1
✅ GPU:          NVIDIA GeForce RTX 3080
✅ VRAM:         10.7 GB

📋 EXAONE Chatbot 설정
베이스 모델:  LGAI-EXAONE/EXAONE-4.0-1.2B
학습 데이터:  c:\Users\user\Documents\fine_tunning\data\refined_law_qa.json
어댑터 경로:  c:\Users\user\Documents\fine_tunning\models\v1_law_adapter
병합 경로:    c:\Users\user\Documents\fine_tunning\models\v1_law
평가 로그:    c:\Users\user\Documents\fine_tunning\evaluation\logs


---
## 2️⃣ HuggingFace 로그인

**왜 하는가?**  
EXAONE 4.0은 LG AI Research의 gated 모델이다. `.env`에 토큰을 넣어두면 매번 입력할 필요가 없다.

> `.env` 파일 예시:
> ```
> HUGGINGFACE_TOKEN=hf_xxxxxxxxxxxxxxxxx
> ```

In [2]:
from dotenv import load_dotenv
from huggingface_hub import login

# 2-1. .env 파일에서 환경변수 로드
#      dotenv_path 명시 → notebooks/ 서 실행해도 루트의 .env를 정확히 찾음
load_dotenv(dotenv_path=os.path.join(ROOT, ".env"))
hf_token = os.getenv("HUGGINGFACE_TOKEN")

# 2-2. 토큰이 있으면 로그인, 없으면 경고만 (이미 CLI로 로그인한 환경 대비)
if hf_token:
    login(token=hf_token)
    print("✅ HuggingFace 로그인 완료")
else:
    print("⚠️  HUGGINGFACE_TOKEN 없음 (이미 로그인 상태면 무시)")

✅ HuggingFace 로그인 완료


---
## 3️⃣ 데이터 로드 (3,000건)

**데이터 포맷: Alpaca**

```json
[
  {
    "instruction": "임차인이 보증금을 못 받았을 때?",
    "input": "",
    "output": "우선 내용증명을 보내고..."
  },
  ...
]
```

HuggingFace `Dataset`으로 감싸면 `.map()`으로 토크나이징을 병렬 처리할 수 있다.

In [3]:
import json
from datasets import Dataset

# 3-1. JSON 파일 전체를 메모리로 로드
#      3,000건 정도면 한 번에 로드해도 문제없음
with open(TRAIN_DATA_PATH, encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"📊 데이터 수: {len(raw_data)}")
print(f"\n첫 샘플:")
print(f"  instruction: {raw_data[0]['instruction'][:80]}...")
print(f"  output:      {raw_data[0]['output'][:80]}...")

# 3-2. list[dict] → HF Dataset 객체
#      이후 .map()으로 벡터화된 전처리가 가능해짐
ds = Dataset.from_list(raw_data)
print(f"\n✅ Dataset: {len(ds)}건")

📊 데이터 수: 3000

첫 샘플:
  instruction: 주택연금 가입자가 사망할 경우, 주택연금 수령 및 담보주택의 상속은 어떻게 처리되나요?...
  output:      주택연금 가입자가 주택연금 수령 중 사망한 경우, 배우자가 있는 경우에는 배우자가 채무인수 및 소유권 이전등기를 완료하면 주택연금을 계속 받을 ...

✅ Dataset: 3000건


---
## 4️⃣ EXAONE 4.0 + QLoRA 로드

**두 단계 작업**:
1. 베이스 모델을 **4-bit 양자화**로 로드 → VRAM 절약
2. `prepare_model_for_kbit_training()` → 양자화 모델을 학습 가능한 상태로 준비

### 왜 4-bit인가?
- 1.2B 파라미터 × fp16 = 약 2.4GB VRAM
- 1.2B 파라미터 × 4-bit = 약 0.6GB VRAM
- 그래디언트/옵티마이저 상태까지 고려하면 학습 시에도 **5GB 이하**로 가능 → RTX 3080/4060에서도 돌릴 수 있음

In [4]:
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
)
from peft import (
    LoraConfig, TaskType, prepare_model_for_kbit_training, get_peft_model,
)

# 4-1. 토크나이저 로드
print("🔄 토크나이저 로드...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

# pad_token 설정 (대부분의 LLM은 pad_token이 없음 → eos로 대체하는 관용 패턴)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# padding_side="right": Causal LM 학습 시 표준.
#   → 라벨 시프트가 자연스럽고, attention mask 처리가 깔끔함
tokenizer.padding_side = "right"
print(f"✅ 어휘: {tokenizer.vocab_size:,}")

# 4-2. 4-bit 양자화 설정 (QLoRA 논문 권장 스펙)
bnb = BitsAndBytesConfig(
    load_in_4bit=True,                         # 가중치를 4-bit로 저장
    bnb_4bit_quant_type="nf4",                 # NormalFloat 4-bit (정규분포 최적화)
    bnb_4bit_compute_dtype=torch.bfloat16,     # 연산은 bf16로 (정확도 유지)
    bnb_4bit_use_double_quant=True,            # 양자화 상수도 양자화 → 추가 VRAM 절감
)

# 4-3. 모델 로드 (첫 실행 시 다운로드 2~4분)
print("\n🔄 EXAONE 4.0 1.2B 4-bit 로드 중...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb,
    device_map="auto",  # GPU 자동 배치
)

# 4-4. 양자화 모델을 학습 가능한 상태로 준비
#   내부에서 하는 일:
#   - Layer Norm 등을 fp32로 cast (수치 안정성)
#   - gradient_checkpointing 활성화 (VRAM 절감)
#   - input embedding의 requires_grad=True 설정
model = prepare_model_for_kbit_training(model)
print(f"✅ VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")

🔄 토크나이저 로드...
✅ 어휘: 102,400

🔄 EXAONE 4.0 1.2B 4-bit 로드 중...


Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]

✅ VRAM: 1.39GB


### LoRA 어댑터 부착

**LoRA(Low-Rank Adaptation)**: 원래 가중치 $W$는 얼려두고, 그 옆에 저차원 행렬 $A \cdot B$만 학습.
- $W_{new} = W + \alpha \cdot A \cdot B$ (A: d×r, B: r×d, r은 아주 작음)
- **r=8**이면 학습 파라미터가 전체의 ~0.1%만 되어도 충분한 성능.

**target_modules**: 어디에 LoRA를 꽂을지.  
일반적으로 attention의 `q_proj`, `k_proj`, `v_proj`, `o_proj`에 부착.

In [5]:
# 5-1. LoRA 설정 객체 생성
#      settings.py의 LORA_CONFIG에서 값 가져와 적용
lora_config = LoraConfig(
    r=LORA_CONFIG["r"],                        # 저차원 랭크 (보통 8)
    lora_alpha=LORA_CONFIG["lora_alpha"],      # 스케일링 계수 (보통 2r)
    lora_dropout=LORA_CONFIG["lora_dropout"],  # 정규화용 dropout (0.05 등)
    bias=LORA_CONFIG["bias"],                  # bias 학습 여부 (보통 "none")
    task_type=TaskType.CAUSAL_LM,              # 다음 토큰 예측 태스크
    target_modules=LORA_CONFIG["target_modules"],  # 부착 위치
)

# 5-2. 모델에 LoRA 어댑터 부착
#      이후 model.parameters() 중 requires_grad=True 인 것만 학습됨
model = get_peft_model(model, lora_config)
print("🎯 QLoRA 부착 완료\n")

# 5-3. 학습 가능한 파라미터 비율 출력
#      보통 "trainable: 0.1% of total" 정도가 나오면 정상
model.print_trainable_parameters()

🎯 QLoRA 부착 완료

trainable params: 15,237,120 || all params: 1,294,628,608 || trainable%: 1.1769


---
## 5️⃣ 토크나이징

**Alpaca → ChatML 변환 + 토큰화**를 한 번에 수행.

```
{"instruction": "...", "output": "..."}
        │
        ▼
[{"role":"system",    "content": SYSTEM_LEGAL},
 {"role":"user",      "content": instruction},
 {"role":"assistant", "content": output}]
        │
        ▼
   apply_chat_template() → 텍스트 프롬프트
        │
        ▼
   tokenizer() → input_ids, attention_mask
        │
        ▼
   labels = input_ids.copy()  (Causal LM은 입력=라벨)
```

> ⚠️ `labels = input_ids.copy()`인 이유: Causal LM은 "다음 토큰 예측"이므로 입력과 정답이 같은 시퀀스. 내부적으로 1칸 시프트되어 loss 계산됨.

In [6]:
def preprocess(sample):
    """Alpaca 포맷 → ChatML → 토큰"""
    # 5-1. ChatML 메시지 구성
    #      system: 도메인 컨텍스트 (법률 어시스턴트 등)
    #      user:   질문 (instruction)
    #      assistant: 정답 (output) — 이게 학습 타겟
    messages = [
        {"role": "system",    "content": SYSTEM_LEGAL},
        {"role": "user",      "content": sample["instruction"]},
        {"role": "assistant", "content": sample["output"]},
    ]

    # 5-2. ChatML 템플릿을 텍스트로 변환 (tokenize=False → 문자열만)
    text = tokenizer.apply_chat_template(messages, tokenize=False)

    # 5-3. 텍스트 → 토큰 ID
    #      max_length로 잘라내고, padding은 collator에서 배치 단위로 처리
    tok = tokenizer(
        text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,  # ← collator가 처리
    )

    # 5-4. Causal LM의 labels는 input_ids와 동일
    #      HF Trainer가 내부에서 1칸 시프트해 loss 계산
    tok["labels"] = tok["input_ids"].copy()
    return tok

# 5-5. Dataset.map()으로 전체 데이터 일괄 전처리
#      remove_columns로 원본 컬럼 삭제 → 메모리 절약
print("🔄 토크나이징...")
tokenized = ds.map(
    preprocess,
    remove_columns=ds.column_names,
    desc="Tokenizing",
)
print(f"✅ {len(tokenized)}건")

🔄 토크나이징...


Tokenizing:   0%|          | 0/3000 [00:00<?, ? examples/s]

✅ 3000건


---
## 6️⃣ 학습 설정

### 주요 하이퍼파라미터 해설

| 항목 | 역할 | 전형적 값 |
|------|------|-----------|
| `num_train_epochs` | 전체 데이터 몇 바퀴 | 3~4 |
| `per_device_train_batch_size` | GPU당 배치 | 2~4 (VRAM 따라) |
| `gradient_accumulation_steps` | 누적 스텝 | 4~8 |
| **실효 배치** | 배치×누적 | 16~32 |
| `learning_rate` | 학습률 | 2e-4 (LoRA 기준) |
| `lr_scheduler_type` | 스케줄러 | cosine |
| `warmup_steps` | 워밍업 | 전체의 3~10% |
| `optim` | 옵티마이저 | `paged_adamw_8bit` (VRAM 절약) |
| `bf16` | 혼합정밀도 | True (Ampere 이상 GPU) |

### 그래디언트 누적이란?
VRAM이 부족해서 배치 32가 안 들어갈 때, 배치 4를 8번 forward+backward 한 뒤 한 번만 optimizer step을 밟는다. **수학적으로 배치 32와 동일한 효과**.

In [7]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

# 6-1. 학습 인자 구성 (settings.py 의 TRAIN_CONFIG 반영)
args = TrainingArguments(
    output_dir=str(ADAPTER_DIR) + "_tmp",  # Trainer 임시 체크포인트 (_tmp로 구분)
    num_train_epochs=TRAIN_CONFIG["num_train_epochs"],
    per_device_train_batch_size=TRAIN_CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=TRAIN_CONFIG["gradient_accumulation_steps"],
    learning_rate=TRAIN_CONFIG["learning_rate"],
    lr_scheduler_type=TRAIN_CONFIG["lr_scheduler_type"],
    warmup_steps=TRAIN_CONFIG["warmup_steps"],
    optim=TRAIN_CONFIG["optim"],             # paged_adamw_8bit 권장
    bf16=TRAIN_CONFIG["bf16"],               # bfloat16 혼합정밀도
    logging_steps=TRAIN_CONFIG["logging_steps"],
    save_strategy="no",                      # 학습 중 저장 안 함 (마지막에 한 번만)
    report_to="none",                        # wandb/tensorboard 비활성
    seed=42,                                  # 재현성
)

# 6-2. 배치 collator
#      pad_to_multiple_of=8 → GPU 텐서코어(8의 배수일 때 가장 빠름) 최적화
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    pad_to_multiple_of=8,
)

# 6-3. 예상 스텝·시간 계산 (디버깅/시간 예측용)
eff_batch = (
    TRAIN_CONFIG["per_device_train_batch_size"]
    * TRAIN_CONFIG["gradient_accumulation_steps"]
)
expected_steps = len(tokenized) // eff_batch * TRAIN_CONFIG["num_train_epochs"]
print(f"📊 실효 배치: {eff_batch}")
print(f"📊 예상 스텝: {expected_steps}")
# 한 스텝당 4~6초 (RTX 3080 기준) 가정
print(f"⏱️  예상 시간: {expected_steps * 4 / 60:.0f}~{expected_steps * 6 / 60:.0f}분")

📊 실효 배치: 8
📊 예상 스텝: 750
⏱️  예상 시간: 50~75분


---
## 7️⃣ 학습 실행 🚀

### 학습 중 모니터링 포인트

| 지표 | 정상 범위 | 이상 징후 |
|------|-----------|-----------|
| `loss` | 첫 스텝 2~3 → 0.5~1.0으로 감소 | 계속 증가하거나 NaN |
| `grad_norm` | 0.1~5 정도 | 수백 이상 (발산 신호) |
| `learning_rate` | warmup 후 설정값 → 0으로 감소 (cosine) | 변화 없음 (스케줄러 오류) |

**loss가 0에 너무 빠르게 떨어지면**? 오버피팅 신호일 수 있음 → epoch 줄이거나 dropout 올려보기.

In [8]:
# 7-1. Trainer 생성
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=collator,
)

# 7-2. 학습 실행 (블로킹 호출)
print("🏋️ 학습 시작!\n")
result = trainer.train()

# 7-3. 결과 요약
print(f"\n✅ 학습 완료!")
print(f"   최종 loss: {result.training_loss:.4f}")
print(f"   소요 시간: {result.metrics['train_runtime']/60:.1f}분")

🏋️ 학습 시작!



c:\Users\user\Documents\fine_tunning\venv\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,2.105165
20,1.229079
30,1.133271
40,1.125433
50,1.084724
60,1.074744
70,1.064620
80,1.108188
90,1.063005
100,1.066581



✅ 학습 완료!
   최종 loss: 0.9809
   소요 시간: 21.3분


---
## 8️⃣ 어댑터 저장

`trainer.save_model()`은 **LoRA 어댑터만** 저장한다 (베이스 모델은 저장 X).
- 어댑터 용량: 약 10~30 MB (원본 모델의 1% 미만)
- 저장되는 파일:
    - `adapter_config.json` — LoRA 설정
    - `adapter_model.safetensors` — LoRA 가중치
    - `tokenizer*` — 토크나이저

베이스 모델 경로는 `adapter_config.json`에 기록되어 있으므로, 나중에 로드할 때 PEFT가 알아서 원본을 함께 불러온다.

In [9]:
# 8-1. LoRA 어댑터만 저장
trainer.save_model(str(ADAPTER_DIR))

# 8-2. 토크나이저도 같이 저장 (독립 사용 가능하도록)
tokenizer.save_pretrained(str(ADAPTER_DIR))

# 8-3. 저장된 파일 목록 확인
print(f"💾 저장: {ADAPTER_DIR}/\n")
for f in sorted(os.listdir(ADAPTER_DIR)):
    size_kb = os.path.getsize(ADAPTER_DIR / f) / 1024
    print(f"   {f:40s} {size_kb:>8.1f} KB")

💾 저장: c:\Users\user\Documents\fine_tunning\models\v1_law_adapter/

   README.md                                     5.1 KB
   adapter_config.json                           1.1 KB
   adapter_model.safetensors                 59574.4 KB
   chat_template.jinja                           5.5 KB
   tokenizer.json                             7724.0 KB
   tokenizer_config.json                         0.4 KB
   training_args.bin                             4.7 KB


---
## 🎉 Stage 3-1 완료!

### 다음 노트북 실행 전 메모리 정리

학습이 끝나면 VRAM이 꽉 차 있다. `02_merge_model.ipynb`에서 fp16으로 베이스 모델을 다시 로드해야 하므로 **반드시 정리**한다.

In [10]:
import gc

# 9-1. 파이썬 참조 제거
del model, trainer

# 9-2. 가비지 컬렉터 수동 실행 (큰 객체 즉시 회수)
gc.collect()

# 9-3. PyTorch가 보유 중인 캐시 메모리 반환
torch.cuda.empty_cache()

print(f"✅ 메모리 정리 완료 (VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB)")
print("\n👉 다음: 02_merge_model.ipynb 실행")

✅ 메모리 정리 완료 (VRAM: 0.86GB)

👉 다음: 02_merge_model.ipynb 실행
